In [32]:
#Import pandas
import pandas as pd

In [34]:
# Central path variable
RAW_PATH = "../data/raw/online_retail_II.xlsx"

In [44]:
# Load each annual sheet separately (source file has two tabs: 2009-2010 and 2010-2011)
# Tagging each with a SourceSheet label BEFORE combining, so we can trace any row back to its
# original sheet later if something looks off (useful for isolating year-specific issues)
sheet_2009_2010 = pd.read_excel(RAW_PATH, sheet_name="Year 2009-2010")
sheet_2010_2011 = pd.read_excel(RAW_PATH, sheet_name="Year 2010-2011")
sheet_2009_2010["SourceSheet"] = "2009-2010"
sheet_2010_2011["SourceSheet"] = "2010-2011"

In [54]:
# Stack both sheets into a single dataframe
# ignore_index=True gives us a clean, continuous 0..N index instead of two overlapping
# per-sheet indexes (which would create duplicate index labels)
# Row count sanity check: len(df) must equal len(sheet1) + len(sheet2), or something went wrong
df = pd.concat([sheet_2009_2010, sheet_2010_2011], ignore_index =True)
print(len(sheet_2009_2010))
print(len(sheet_2010_2011))
print(len(df))

525461
541910
1067371


In [56]:
# Validate that both sheets share identical column names before trusting the concat
# (mismatched names, e.g. 'CustomerID' vs 'Customer ID', would silently create extra columns
# instead of merging cleanly) then print the final combined column list for reference
print(list(sheet_2009_2010.columns) == list(sheet_2010_2011.columns))
print(df.columns.tolist())

True
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'SourceSheet']


In [58]:
# Quick structural overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
 8   SourceSheet  1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 73.3+ MB


In [60]:
# Distinct value count per column sanity checks scale (#invoices, #products, #customers)
# and flags anomalies like Description having more unique values than StockCode
df.nunique()

Invoice        53628
StockCode       5305
Description     5698
Quantity        1057
InvoiceDate    47635
Price           2807
Customer ID     5942
Country           43
SourceSheet        2
dtype: int64

In [70]:
# Exact missing-value counts per column confirms Description (4,382) and Customer ID (243,007)
# gaps found in .info(), now as precise numbers we can quote directly 
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
SourceSheet         0
dtype: int64

In [66]:
# Full descriptive stats numeric columns get mean/std/min/max/percentiles, text columns get
# count/unique/top/freq (include='all'). .T transposes so columns become rows easier to scan.
# This is where the red flags first appear negative min Price, huge min/max Quantity
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Invoice,1067371.0,53628.0,537434.0,1350.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,1067371,5305,85123A,5829,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,1062989,5698,WHITE HANGING HEART T-LIGHT HOLDER,5918,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,1067371.0,NaN,NaN,NaN,9.938898,-80995.0,1.0,3.0,10.0,80995.0,172.705794
InvoiceDate,1067371,NaN,NaN,NaN,2011-01-02 21:13:55.394028544,2009-12-01 07:45:00,2010-07-09 09:46:00,2010-12-07 15:28:00,2011-07-22 10:23:00,2011-12-09 12:50:00,NaN
Price,1067371.0,NaN,NaN,NaN,4.649388,-53594.36,1.25,2.1,4.15,38970.0,123.553059
Customer ID,824364.0,NaN,NaN,NaN,15324.638504,12346.0,13975.0,15255.0,16797.0,18287.0,1697.46445
Country,1067371,43,United Kingdom,981330,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SourceSheet,1067371,2,2010-2011,541910,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [72]:
# Isolate the exact row driving the extreme negative Price minimum, to inspect it directly
# rather than trust the summary number alone turned out to be a 'bad debt' adjustment, not a sale
df[df["Price"] == df["Price"].min()]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,2009-2010


In [78]:
# Frequency count of Invoice's first character — reveals every invoice-numbering convention
# in use (5xxxxx / 4xxxxx = normal sales, C = cancellations, A = accounting adjustments)
df["Invoice"].astype(str).str[0].value_counts()

Invoice
5    939382
4    108489
C     19494
A         6
Name: count, dtype: int64

In [86]:
# Inspect all rows with 'A' prefixed invoices directly confirms these are bad debt
# accounting entries (StockCode 'B'), not real customer transactions
df[df["Invoice"].astype(str).str[0] =="A"]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,2009-2010
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom,2009-2010
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom,2009-2010
825443,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,2010-2011
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,2010-2011
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,2010-2011


In [96]:
# Row(s) at the minimum Quantity value investigating the extreme outlier found in .describe()
df[df["Quantity"]==df["Quantity"].min()]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom,2010-2011


In [98]:
# Row(s) at the maximum Quantity value pairing with the minimum above revealed this is a
# large order (581483) immediately reversed by its own cancellation (C581484), not a data error
df[df["Quantity"]==df["Quantity"].max()]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,2010-2011


In [102]:
# Filter to all cancellation invoices (C-prefix) used below to test whether 'C-invoice' and
# 'negative quantity' are fully interchangeable signals, or if there are exceptions
df[df["Invoice"].astype(str).str[0]=="C"]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,2009-2010
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,2009-2010
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,2009-2010
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,2009-2010
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,2009-2010
...,...,...,...,...,...,...,...,...,...
1065910,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom,2010-2011
1067002,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom,2010-2011
1067176,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom,2010-2011
1067177,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom,2010-2011


In [108]:
# Count how many rows are involved in exact duplicates
# keep=False flags every row with an exact match elsewhere (not just the 2nd+ occurrence),
# so we can inspect the full duplicate sets
# Then pull the actual duplicate rows for side by side comparison
# Sorted by Invoice/StockCode so matching rows sit next to each other for visual comparison
dupe_mask =df.duplicated(keep=False)
print(dupe_mask.sum())
#Acutal duplicate rows for comparision
df[dupe_mask].sort_values(["Invoice","StockCode"]).head(20)

23430


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,2009-2010
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,2009-2010
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom,2009-2010


In [130]:
# Build a Revenue column (Quantity x Price) needed for every $ impact question from here on
# Isolate rows lacking a Customer ID to quantify their revenue contribution
# Sum Revenue within just that missing-ID subset
# Key finding: missing IDs are ~22.8% of rows but only ~13.7% of revenue — informs the
# decision to keep these rows for revenue analysis but exclude them from customer-level work
df["Revenue"] = df["Quantity"] * df["Price"]
total_revenue = df["Revenue"].sum()
print("Total Revenue:", total_revenue)
#Rows with missing Customer ID
missing_customer_rows = df[df["Customer ID"].isnull()]
#Revenue from those rows
missing_customer_revenue = missing_customer_rows["Revenue"].sum()
print("Revenue from missing Customer ID rows:", missing_customer_revenue)
# What percentage of total revenue that represents
pct = (missing_customer_revenue / total_revenue) * 100
print("Percentage of total revenue:", pct)

Total Revenue: 19287250.56799999
Revenue from missing Customer ID rows: 2638958.18
Percentage of total revenue: 13.682396932087194


In [138]:
# Describe Quantity within C-invoices only — max came back positive (not negative as expected),
# revealing C-invoices aren't a perfect 1:1 match with 'cancellation = negative quantity'
df[df["Invoice"].astype(str).str[0] =="C"]["Quantity"].describe()

count    19494.000000
mean       -25.186827
std        805.104908
min     -80995.000000
25%         -6.000000
50%         -2.000000
75%         -1.000000
max          1.000000
Name: Quantity, dtype: float64

In [140]:
# Drop the accidental lowercase duplicate 'revenue' column, keeping the canonical 'Revenue'
df = df.drop(columns=["revenue"])

In [146]:
# Find the C-invoice exception with positive Quantity — turned out to be a manual accounting
# entry (StockCode 'M', 'Manual', no customer), not a genuine sale reversal
df[(df["Invoice"].astype(str).str[0]=="C") & (df["Quantity"] >0)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,Revenue
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom,2009-2010,373.57


In [148]:
# Reverse check: negative-Quantity rows that AREN'T C-invoices  revealed a separate category
# entirely: stock write offs (Price=0, no customer, notes like 'lost'/'missing'/'smashed')
df[(df["Invoice"].astype(str).str[0].ne("C")) &(df["Quantity"] <0)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,Revenue
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,2009-2010,-0.0
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,2009-2010,-0.0
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,2009-2010,-0.0
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,2009-2010,-0.0
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,2009-2010,-0.0
...,...,...,...,...,...,...,...,...,...,...
1060794,581210,23395,check,-26,2011-12-07 18:36:00,0.0,NaN,United Kingdom,2010-2011,-0.0
1060796,581212,22578,lost,-1050,2011-12-07 18:38:00,0.0,NaN,United Kingdom,2010-2011,-0.0
1060797,581213,22576,check,-30,2011-12-07 18:38:00,0.0,NaN,United Kingdom,2010-2011,-0.0
1062371,581226,23090,missing,-338,2011-12-08 09:56:00,0.0,NaN,United Kingdom,2010-2011,-0.0


In [154]:
# Count how many of those non-C negative-quantity rows also have Price=0 confirms all 3,457
# stock write-off rows share this exact signature (fully explained, no loose ends)
df[(df["Invoice"].astype(str).str[0].ne("C")) &(df["Quantity"] <0) &(df["Price"] ==0)].shape[0]

3457

In [170]:
# Overlap check rows where Quantity<=0 AND Price=0 at the same time
# (the standalone Quantity<=0 total is calculated in the next cell)
((df["Quantity"] <=0) & (df["Price"]==0)).sum()

3457

In [172]:
# Overlap check: rows where Quantity<=0 AND Price=0 at the same time
# (the standalone Quantity<=0 total is calculated in the next cell)
(df["Quantity"] <=0).sum()

22950

In [174]:
# Total rows with Price <= 0 across the whole dataset (6,207) larger than the write-off +
# bad-debt count alone, signaling a third, not-yet-identified category exists
(df["Price"]<=0).sum()

6207

In [176]:
# Isolate the unexplained group: positive Quantity but Price <= 0  revealed free samples,
# promotional giveaways, postage lines (StockCode 'DOT'), and rows with missing descriptions
df[(df["Quantity"]>0) & (df["Price"] <=0)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,Revenue
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom,2009-2010,0.0
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom,2009-2010,0.0
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom,2009-2010,0.0
5904,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00,0.0,NaN,United Kingdom,2009-2010,0.0
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom,2009-2010,0.0
...,...,...,...,...,...,...,...,...,...,...
1060795,581211,22142,check,14,2011-12-07 18:36:00,0.0,NaN,United Kingdom,2010-2011,0.0
1062442,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom,2010-2011,0.0
1063965,581406,46000M,POLYESTER FILLER PAD 45x45cm,240,2011-12-08 13:58:00,0.0,NaN,United Kingdom,2010-2011,0.0
1063966,581406,46000S,POLYESTER FILLER PAD 40x40cm,300,2011-12-08 13:58:00,0.0,NaN,United Kingdom,2010-2011,0.0


In [180]:
# Count of that positive-quantity / zero-or-negative-price group (2,750 rows) combined with
# the write-offs and bad debt, this fully reconciles the total Price<=0 count
((df["Quantity"]>0) &(df["Price"]<=0)).sum()

2750

In [186]:
# Rows with Price exactly 0 (6,202) split out from negative prices for a precise reconciliation
(df["Price"] ==0).sum()

6202

In [188]:
# Rows with Price strictly negative (5) together with the 6,202 zero-price rows, this exactly
# reconciles to the 6,207 total Price<=0 found earlier
(df["Price"]<0).sum()

5

In [190]:
#Quick preview of full dataframe
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-2010,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-2010,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-2010,30.00
...,...,...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,2010-2011,12.60
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,2010-2011,16.60
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,2010-2011,16.60
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France,2010-2011,14.85


In [194]:
# Group by StockCode and count DISTINCT Description values per group the key diagnostic for
# finding product codes that carry more than one description text across different rows
description_counts = df.groupby("StockCode")["Description"].nunique()
print(description_counts)

StockCode
10002           1
10080           2
10109           1
10120           2
10125           1
               ..
gift_0001_60    0
gift_0001_70    1
gift_0001_80    1
gift_0001_90    0
m               1
Name: Description, Length: 5305, dtype: int64


In [196]:
# Filter down to only StockCodes with more than one distinct Description these are the
# actual naming inconsistencies worth investigating (1,232 out of 5,305 codes)
inconsistent_stockcodes = description_counts[description_counts > 1]
print(inconsistent_stockcodes)

StockCode
10080           2
10120           2
10133           2
16008           2
16011           2
               ..
DCGS0068        2
DCGS0069        2
DCGSSBOY        2
DCGSSGIRL       2
gift_0001_20    2
Name: Description, Length: 1232, dtype: int64


In [198]:
# How many inconsistent StockCodes are there in total scale of the issue before digging in
inconsistent_stockcodes.shape[0]

1232

In [202]:
# Inspect one inconsistent code directly (10080) revealed the real product name plus a
# stock adjustment note ('check') and a missing value riding on the same code
df[df["StockCode"].astype(str).str.strip() =="10080"]["Description"].unique()

array(['GROOVY CACTUS INFLATABLE', nan, 'check'], dtype=object)

In [206]:
# Second example (16008) same pattern confirmed: real product name + 'check' note,
# strengthening the theory that most 'inconsistencies' are operational notes, not naming conflicts
df[df["StockCode"].astype(str).str.strip() == "16008"]["Description"].unique()

array(['SMALL FOLDING SCISSOR(POINTED EDGE)', 'check'], dtype=object)

In [208]:
# Test how much of the inconsistency is explained by known stock-adjustment vocabulary
# 357 of the 1,232 codes have at least one row using one of these note-words as the Description
note_words = ["check", "lost", "missing", "smashed", "short", "damaged", "found", "wrong", "?", "thrown away", "faulty"]
df[df["Description"].isin(note_words)]["StockCode"].nunique()

357

In [210]:
# Check an inconsistent code NOT explained by the note-word list (10120) revealed a new
# category: ad-hoc operational annotations ('Zebra invcing error') rather than stock-check notes
df[df["StockCode"].astype(str).str.strip() == "10120"]["Description"].unique()

array(['DOGGY RUBBER', 'Zebra invcing error'], dtype=object)

In [212]:
# Another unexplained code (16011) revealed a third, distinct cause pure whitespace/
# formatting noise (' ANIMAL STICKERS' vs 'ANIMAL STICKERS'), not a real inconsistency at all
df[df["StockCode"].astype(str).str.strip() == "16011"]["Description"].unique()

array([' ANIMAL STICKERS', 'ANIMAL STICKERS'], dtype=object)

In [214]:
# Full list of unique StockCode values starting point for identifying non-product
# (administrative/service) codes that don't follow the normal digit-leading product pattern
df["StockCode"].unique()

array([85048, '79323P', '79323W', ..., 23609, 23617, 23843], dtype=object)

In [224]:
# Boolean check does each StockCode's first character look like a digit? (Series of True/False,
# one per row)  building block for isolating the non-standard codes in the next cell
df["StockCode"].astype(str).str[0].str.isdigit()

0           True
1           True
2           True
3           True
4           True
           ...  
1067366     True
1067367     True
1067368     True
1067369     True
1067370    False
Name: StockCode, Length: 1067371, dtype: bool

In [228]:
# Filter to rows where StockCode does NOT start with a digit (~ inverts the boolean check above),
# then count how often each such code appears — surfaces POST, DOT, M, C2, D, B, and the DCGS batch
non_digit_stockcodes = df[~df["StockCode"].astype(str).str[0].str.isdigit()]
non_digit_stockcodes["StockCode"].value_counts()

StockCode
POST        2122
DOT         1446
M           1421
C2           282
D            177
            ... 
DCGS0027       1
DCGS0016       1
DCGS0006       1
DCGS0044       1
DCGS0036       1
Name: count, Length: 62, dtype: int64

In [240]:
# Confirm what 'POST' represents 'POSTAGE', a shipping charge, not a physical product
df[df["StockCode"] == "POST"]["Description"].unique()

array(['POSTAGE', nan], dtype=object)

In [234]:
# Confirm what 'C2' represents 'CARRIAGE', another shipping/logistics charge code
df[df["StockCode"] == "C2"]["Description"].unique()

array(['CARRIAGE', nan], dtype=object)

In [238]:
# Confirm what 'D' represents 'Discount', an adjustment code, not a product
df[df["StockCode"]=="D"]["Description"].unique()

array(['Discount'], dtype=object)

In [246]:
# Spot check two codes from the long-tail 'DCGS...' batch mixed results (missing description,
# an internal 'update' note, and one genuine product name), suggesting a miscellaneous/
# discontinued product batch rather than one uniform category
df[df["StockCode"] == "DCGS0016"]["Description"].unique()
df[df["StockCode"] == "DCGSSBOY"]["Description"].unique()

array([nan, 'update', 'BOYS PARTY BAG'], dtype=object)

In [248]:
# Earliest InvoiceDate in the dataset validating the data covers the expected start date
df["InvoiceDate"].min()

Timestamp('2009-12-01 07:45:00')

In [250]:
# Latest InvoiceDate in the dataset validating the data covers the expected end date,
# with no stray/erroneous future or past dates
df["InvoiceDate"].max()

Timestamp('2011-12-09 12:50:00')

In [256]:
# Least frequent countries (tail of the sorted value_counts) quick scan for anything unusual
# hiding at the bottom of the distribution (e.g. only 10 rows for Saudi Arabia)
df["Country"].value_counts().tail(5)

Country
West Indies       54
Bermuda           34
Nigeria           32
Czech Republic    30
Saudi Arabia      10
Name: count, dtype: int64

In [254]:
# Total distinct country count cross check against the row-count reconciliation below
# to make sure every country is genuinely accounted for
df["Country"].nunique()

43

In [264]:
# Sum of all country counts should equal total row count (1,067,371) confirms no rows
# are silently missing from the country breakdown (e.g. due to hidden nulls or whitespace)
df["Country"].value_counts().sum()

1067371

## Profiling Summary — Key Findings

**Dataset overview**
- Combined 1,067,371 rows from two sheets (Year 2009-2010: 525,461 / Year 2010-2011: 541,910), row counts and columns validated to match cleanly.
- Date range confirmed: 2009-12-01 to 2011-12-09 matches expected window, no anomalies.

**Missing data**
- `Customer ID`: 243,007 rows missing (22.8%), representing **13.7%** of total revenue (£2.64M of £19.29M). Decision: keep for revenue/product-level
    analysis, exclude only from customer-level analysis (RFM, cohorts).
- `Description`: 4,382 rows missing largely tied to non-sales rows (see below), not spread randomly across genuine products.

**Non-sales transaction types identified**
Three distinct categories found and quantified, all requiring exclusion from revenue/sales KPIs:
1. **Cancellations** 19,493 `C`-prefixed invoices with negative quantity (genuine order cancellations). One exception: `C496350` is a manual accounting
    entry (StockCode `M`, positive quantity, no customer).
2. **Stock write-offs** 3,457 rows, ordinary invoice numbers, negative quantity, Price = 0, descriptive notes ("lost", "missing", "smashed", "check")
    in place of product name.
3. **Bad debt adjustments**  6 rows, StockCode `B`, "Adjust bad debt," no customer attached.
4. **Free/promotional movements** 2,750 rows with positive quantity but Price ≤ 0 (samples, promotions, postage, stock corrections).

Combined: `Quantity ≤ 0` = 22,950 rows (fully reconciled); `Price ≤ 0` = 6,207 rows (fully reconciled).

**Non-product StockCodes**
Identified and quantified: `POST` (2,122, Postage), `DOT` (1,446, Dotcom Postage), `M` (1,421, Manual), `C2` (282, Carriage), `D` (177, Discount),
    `B` (6, Bad debt), plus a long tail of `DCGS...`-prefixed codes (miscellaneous/discontinued batch, mixed real products and internal notes).
All excluded from product-level sales analysis; retained separately if a full revenue reconciliation (incl. fees/charges) is needed.

**StockCode / Description inconsistencies**
1,232 of 5,305 StockCodes (23%) map to more than one Description. Investigated and found not to be genuine product-naming conflicts — driven by 
three causes: stock-adjustment notes (357 codes confirmed), operational/error annotations (e.g. "Zebra invcing error"), and whitespace/formatting noise.
Cleaning approach: build a canonical StockCode→Description lookup using the most frequent, whitespace-stripped, non-note description per code.

**Duplicate rows**
23,430 rows (2.2%) are exact duplicates. Investigated directly — pattern suggests the source system logs repeated identical purchases as separate lines
rather than consolidating quantity, not an export error. Decision: retain, documented, since dropping risks understating genuine sales.

**Country distribution**
43 distinct countries. UK dominates (~92%, 981,330 rows). Two non-standard entries flagged: `Unspecified` (756 rows) and `European Community` (61 rows) treat as ambiguous/excluded in country-level analysis. `EIRE` = Ireland, `RSA` = South Africa — worth standardizing if joining against ISO country
reference data.

---
**Next step:** these findings define the transaction-type classification and exclusion rules for `02_data_cleaning.ipynb`.